Billy York

5/14/26


# Content-Based Filtering

In [ ]:
import os
os.environ['POLARS_MAX_THREADS'] = '8'
os.environ['POLARS_ENGINE_AFFINITY'] = 'streaming'
os.environ['_RJEM_MALLOC_CONF'] = 'abort_conf:true,dirty_decay_ms:0,muzzy_decay_ms:0'

In [ ]:
import polars as pl
from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import seaborn as sns

In [ ]:
def normalize_ids(df: pl.DataFrame, id: str):
    uniq_ids = set(df[id].unique())
    ids_to_indices = { orig_id : new_id for new_id, orig_id in enumerate(uniq_ids) }
    new_ids = df.with_columns(pl.col(id).replace(ids_to_indices).cast(pl.Int64))
    indices_to_ids = { new_id : orig_id for orig_id, new_id in ids_to_indices.items()}
    return new_ids, indices_to_ids

## Exploring and Joining the Data

In [ ]:
# artists = pl.read_csv('artists_reduced.csv')
# designers = pl.read_csv('designers_reduced.csv')
# games = pl.read_csv('games.csv')
# mechanics = pl.read_csv('mechanics.csv')
# publishers = pl.read_csv('publishers_reduced.csv')
# ratings = pl.read_csv('ratings_distribution.csv')
# subcategories = pl.read_csv('subcategories.csv')
# themes = pl.read_csv('themes.csv')
# user_ratings = pl.read_csv('user_ratings.csv')

# artists.write_ipc('artists_reduced.feather')
# designers.write_ipc('designers_reduced.feather')
# games.write_ipc('games.feather')
# mechanics.write_ipc('mechanics.feather')
# publishers.write_ipc('publishers_reduced.feather')
# ratings.write_ipc('ratings_distribution.feather')
# subcategories.write_ipc('subcategories.feather')
# themes.write_ipc('themes.feather')
# user_ratings.write_ipc('user_ratings.feather')

In [ ]:
artists = pl.read_ipc('artists_reduced.feather')
designers = pl.read_ipc('designers_reduced.feather')
games = pl.read_ipc('games.feather')
mechanics = pl.read_ipc('mechanics.feather')
publishers = pl.read_ipc('publishers_reduced.feather')
ratings = pl.read_ipc('ratings_distribution.feather')
subcategories = pl.read_ipc('subcategories.feather')
themes = pl.read_ipc('themes.feather')
user_ratings = pl.read_ipc('user_ratings.feather')

In [ ]:
artists.head()

In [ ]:
designers.head()

In [ ]:
games.head()

In [ ]:
mechanics.head()

In [ ]:
publishers.head()

In [ ]:
ratings.head()

In [ ]:
subcategories.head()

In [ ]:
themes.head()

In [ ]:
user_ratings.head()

In [ ]:
games = games.lazy().join(other=artists.lazy(), on='BGGId').collect()
games = games.lazy().join(other=designers.lazy(), on='BGGId').collect()
games = games.lazy().join(other=mechanics.lazy(), on='BGGId').collect()
games = games.lazy().join(other=publishers.lazy(), on='BGGId').collect()
games = games.lazy().join(other=ratings.lazy(), on='BGGId')
games = games.lazy().join(other=subcategories.lazy(), on='BGGId').collect()
games = games.lazy().join(other=themes.lazy(), on='BGGId').collect()

## Feature Engineering

Ideas:
- Games
    - Same Artist & Designer
    - Number of Mechanics
    - Number of Votes > Number Owned (Hype)
    - Mfg Age vs Com Age
    - Publisher Avg Rating vs Game Avg Rating
    - Mfg Playtime vs Com Playtimes
    - Range of Player Counts Deemed Good (len(pl.col('good_player_counts')))

Feature to show if the game's art and design are from the same person.

The idea is someone might prefer games with a "more holistic vision".

Whether or not a game having the same artists and designer is better or worse is beyond the scope of this experiment, this is simply to see if there is a trend in preferences for having the same artist and designer

In [ ]:
art_des = []
multi = []

artists.drop_in_place('BGGId')
designers.drop_in_place('BGGId')

for artist in artists.columns:
    if (artist in designers.columns):
        multi.append(artist)

for game in games.iter_rows(named=True):
    same = 0
    for artist in multi:
        if (game[artist] == 1 and game[f'{artist}_right'] == 1):
            same = 1
    art_des.append(same)
games.insert_column(len(games.columns), pl.Series('SameArtAndDes', art_des))
games.filter(pl.col('SameArtAndDes') == 1).head()

In [ ]:
games = games.with_columns(NumArtists=pl.sum_horizontal(artists.columns))
for artist in artists.columns:
    games.drop_in_place(artist)

In [ ]:
designer_columns = designers.columns
new_designer_columns = []
for designer in designer_columns:
    same = False
    for artist in multi:
        if artist in designer:
            same = True
        else:
            same = same or False
    if same:
        new_designer_columns.append(f'{designer}_right')
    else:
        new_designer_columns.append(designer)
games = games.with_columns(NumDesigners=pl.sum_horizontal(new_designer_columns))
for designer in new_designer_columns:
    games.drop_in_place(designer)

Feature to simply count the number of mechanics in a game.

This may be iterated on to compare to community weight, the score for complexity of a game which should increase with the number of mechanics

In [ ]:
games = games.with_columns(NumMechanics=pl.sum_horizontal([x for x in mechanics.columns if x != 'BGGId']))
games.select(mechanics.columns + ['NumMechanics'])

Feature to mark if a game has more ratings than owners.

There is a trend, particularly among kickstarted games, though the use of that feature has been omitted, where people will vote on a game they want to succeed (or fail) even if they have not played it. The number of plays feature is not being used because ancedotally speaking, it will always be significantly less than the actual number of plays a game sees. That is to say, the number of plays feature is extremely subject to voluntary response bias. 

In [ ]:
games = games.with_columns(OverHyped=pl.col('NumUserRatings').gt(pl.col('NumOwned')).cast(pl.Int64))
games.select(['NumUserRatings', 'NumOwned', 'OverHyped'])

Feature to show if there is a difference between the Manufacturers Age Recommendation and the Community reported Age Recommendation.

Ideally the community recommendation will be lower, particularly for "Junior" or children games.

In [ ]:
games = games.with_columns(AgeRecDiff=pl.col('MfgAgeRec') - pl.col('ComAgeRec'))
games.select(['MfgAgeRec', 'ComAgeRec', 'AgeRecDiff'])

Feature to show if a game is better than its publisher(s) average showing.

If a game has a higher rating than what a publisher usually puts out, then it should be an easy recommendation. Similarly, if a game is a worse showing for a publisher, it likely shouldn't be recommended, even to fans of that publisher.

Because games can have multiple publishers and there is not an easy way to show the difference between "primary" publishers and publishers who are publishing outside the target market, they have all simply been combined into a single average. 

In [ ]:
publisherNames = publishers.columns
publisherNames.remove('BGGId')
publisherNames.remove('Low-Exp Publisher')
publisherRatings = {}
publisherAvgRatings = {}
gamePublisherRatios = []

for publisher in publisherNames:
    publisherRatings[publisher] = []

for game in games.iter_rows(named=True):
    for publisher in publisherNames:
        if game[publisher] == 1:
            publisherRatings[publisher].append(game['AvgRating'])

for publisher in publisherNames:
    publisherAvgRatings[publisher] = sum(publisherRatings[publisher]) / len(publisherRatings[publisher])



for game in games.iter_rows(named=True):
    publisherRatios = []
    for publisher in publisherNames:
        if game[publisher] == 1:
            if publisherAvgRatings[publisher] > game['AvgRating']:
                publisherRatios.append(1)
            elif publisherAvgRatings[publisher] == game['AvgRating']:
                publisherRatios.append(0)
            else:
                publisherRatios.append(-1)
    if len(publisherRatios) > 0:
        gamePublisherRatios.append(sum(publisherRatios) / len(publisherRatios))
    else:
        gamePublisherRatios.append(0)

games.insert_column(len(games.columns), pl.Series('PublisherAvgRatingRatio', gamePublisherRatios))
for publiser in publisherNames:
    games.drop_in_place(publiser)
games.filter(pl.col('PublisherAvgRatingRatio') > 0).select(['Name', 'PublisherAvgRatingRatio'])

Feature to show if the Manufacture Recommended Playtime is different from the average community reported playtime.

Generally speaking, a game going past the time on the box is common and seen as a negative thing as people can't properly plan around it. The values for the comparisons reflect the same, longer playtimes are negative and shorter playtimes are positive. 

This feature could be a check for inside the range instead of compared to the middle of the community max and min. 

In [ ]:
timeDiscrepencies = []
for game in games.iter_rows(named=True):
    avgComTime = game['ComMinPlaytime'] + ((game['ComMaxPlaytime'] - game['ComMinPlaytime']) / 2)
    if game['MfgPlaytime'] > avgComTime:
        timeDiscrepencies.append(1)
    elif game['MfgPlaytime'] == avgComTime:
        timeDiscrepencies.append(0)
    else:
        timeDiscrepencies.append(-1)
games.insert_column(len(games.columns), pl.Series('MfgTimeAccuracy', timeDiscrepencies))
games.select(['MfgPlaytime', 'ComMinPlaytime', 'ComMaxPlaytime', 'MfgTimeAccuracy'])

Feature to give the ratio of Number of Players the game _can_ play vs the Number of Players the game _should_ play.

There are few games which feel good to play at each player count and typically they are balanced around a particular player count, so ones on the edges of the range could have been added simply to give the game a wider market.

In [ ]:
playerCountRatio = []
for game in games.iter_rows(named=True):
    if game['MaxPlayers'] < game['MinPlayers']:
        playerCountRatio.append(0) # Bad/Absent Data or Solo Game
    else:
        playerCountRatio.append(len(game['GoodPlayers'][1:-1].split(',')) / (1 + (game['MaxPlayers'] - game['MinPlayers'])))
games.insert_column(len(games.columns), pl.Series('PlayerCountRatio', playerCountRatio))
games.select(['GoodPlayers', 'MaxPlayers', 'MinPlayers', 'PlayerCountRatio'])

In [ ]:
playerCounts = set([])
playerCountSeries = {}
for game in games.iter_rows(named=True):
    for count in game['GoodPlayers'][1:-1].split(','):
        playerCounts.add(count.strip('\' +'))

playerCounts.remove('')

for count in playerCounts:
    playerCountSeries[f'GoodAt{count}Players'] = []

for game in games.iter_rows(named=True):
    for count in playerCountSeries.keys():
        gameCounts = game['GoodPlayers'][1:-1].split(',')
        newCounts = list(map(lambda x: x.strip('\' +'), gameCounts))
        if count[7:-7] in newCounts:
            playerCountSeries[count].append(1)
        else:
            playerCountSeries[count].append(0)

for count in playerCountSeries.keys():
    games.insert_column(len(games.columns), pl.Series(count, playerCountSeries[count]))

games.drop_in_place('GoodPlayers')

games.filter(pl.col('GoodAt5Players').eq(1)).select(['Name', 'GoodAt5Players'])

In [ ]:
games.drop_in_place('Name')
games.drop_in_place('Description')
games.drop_in_place('ImagePath')
games = games.to_dummies('Family')

- Users
    - Average Relative Rating Given (Hater or Glazer)
    - Avg relative number of ratings on games rated (Mainstream or hipster)
    - Average relative weight (Wargamer or Casual)
    - Average relative Age of Games Rated (Old School or Cult of the New)
    - Average relative best players (Solo Gamer or Party Gamer)

These features ended up all as relative to averages, mostly because I was unsure how I would enocode something like "favorite game type" without adding more one-hot encoded features which would significantly grow the size of the networks. 

Feature to determine if the User is rating a game better or worse than their average rating. My expectation is a higher differential would lead to a higher predicted rating for similar games.

In [ ]:
user_avg_ratings = user_ratings.group_by(pl.col('Username')).agg(AvgRating=pl.col('Rating').mean())
user_avg_ratings = user_avg_ratings.with_columns(pl.col('AvgRating').mul(-1))
user_avg_rating_diffs = user_avg_ratings.join(user_ratings, on='Username').with_columns(RatingDiff=pl.sum_horizontal('AvgRating', 'Rating'))
user_avg_rating_diffs.head()

All subcultures have their snobs, so on an aggregation website like BoardGameGeek, there will be users who will probably dislike something just because it is popular (See most of Gloomhaven's negative reviews around when it released). So this feature should determine if the User prefers games that are mainstream or more unknown and marks the games accordingly.

In [ ]:
cols = games.columns.copy()
cols.remove('BGGId')
cols.remove('NumUserRatings')
user_avg_num_ratings = user_ratings.join(games.drop(cols), on='BGGId')
user_avg_num_ratings = user_avg_num_ratings.group_by(pl.col('Username')).agg(AvgNumRatings=pl.col('NumUserRatings').mean())
user_avg_num_ratings = user_avg_num_ratings.with_columns(pl.col('AvgNumRatings').mul(-1))
user_avg_num_rating_diffs = user_avg_num_ratings.join(user_ratings, on='Username').join(games.drop(cols), on='BGGId').with_columns(NumRatingDiff=pl.sum_horizontal('AvgNumRatings', 'NumUserRatings'))
user_avg_num_rating_diffs.head()

People have different preferences of "weight" for a game, and while attempting to put all types of game complexities into a single number is likely a fools errand, someone who is seeking out games probably uses the number to determine if the game is for them or not. So this feature should determine if the User prefers heavy weight or light weight games and marks the games accordingly.

In [ ]:
cols = games.columns.copy()
cols.remove('BGGId')
cols.remove('GameWeight')
user_avg_weight = user_ratings.join(games.drop(cols), on='BGGId')
user_avg_weight = user_avg_weight.group_by(pl.col('Username')).agg(AvgWeight=pl.col('GameWeight').mean())
user_avg_weight = user_avg_weight.with_columns(pl.col('AvgWeight').mul(-1))
user_avg_weight_diffs = user_avg_weight.join(user_ratings, on='Username').join(games.drop(cols), on='BGGId').with_columns(GameWeightDiff=pl.sum_horizontal('AvgWeight', 'GameWeight'))
user_avg_weight_diffs.head()

There has been a noted phenomena that people are preferring to play newer games and Board Games are shifting into more common media trends rather than its previous status of having "shelf staples" where regardless of their age, certain games were always must plays. So this feature should determine if a User prefers older or newer games and marks the games accordingly.

In [ ]:
cols = games.columns.copy()
cols.remove('BGGId')
cols.remove('YearPublished')
user_avg_age = user_ratings.join(games.drop(cols), on='BGGId')
user_avg_age = user_avg_age.with_columns(AvgAge=2026 - pl.col('YearPublished'))
user_avg_age = user_avg_age.group_by(pl.col('Username')).agg(pl.col('AvgAge').mean())
user_avg_age = user_avg_age.with_columns(pl.col('AvgAge').mul(-1))
user_avg_age_diffs = user_avg_age.join(user_ratings, on='Username').join(games.drop(cols), on='BGGId').with_columns(Age=2026 - pl.col('YearPublished')).with_columns(AgeDiff=pl.sum_horizontal('AvgAge', 'Age'))
user_avg_age_diffs.head()

A lot of players have a core group they play board games with, so they'll seek games that can accomodate that size. My group is 5 people and it's tough to see so many good games be only up to 4 players. Additionally, Covid brought in new trends of Solo Games and Duel variants of popular games, the latter being something I have had a lot of fun exploring with my partner. So this feature attempts to find if a User prefers games for a particular group size and marks games as such. 

In [ ]:
cols = games.columns.copy()
cols.remove('BGGId')
cols.remove('BestPlayers')
user_avg_players = user_ratings.join(games.drop(cols), on='BGGId')
user_avg_players = user_avg_players.group_by(pl.col('Username')).agg(AvgPlayers=pl.col('BestPlayers').mean())
user_avg_players = user_avg_players.with_columns(pl.col('AvgPlayers').mul(-1))
user_avg_players_diffs = user_avg_players.join(user_ratings, on='Username').join(games.drop(cols), on='BGGId').with_columns(BestPlayerDiff=pl.sum_horizontal('AvgPlayers', 'BestPlayers'))
user_avg_players_diffs.head()

In [ ]:
ratings_df = user_ratings.join(games, on='BGGId')
ratings_df = ratings_df.with_columns(
    RatingDiff=user_avg_rating_diffs['RatingDiff'],
    NumRatingDiff=user_avg_num_rating_diffs['NumRatingDiff'],
    GameWeightDiff=user_avg_weight_diffs['GameWeightDiff'],
    AgeDiff=user_avg_age_diffs['AgeDiff'],
    BestPlayerDiff=user_avg_players_diffs['BestPlayerDiff']
)
del user_avg_rating_diffs
del user_avg_num_rating_diffs
del user_avg_weight_diffs
del user_avg_age_diffs
del user_avg_players_diffs
del artists
del designers
del games
del mechanics
del publishers
del ratings
del subcategories
del themes
del user_ratings
ratings_df.head()

## Experimental Design

In [ ]:
gss = GroupShuffleSplit(n_splits=5, random_state=42)

In [ ]:
uniq_item_ids = set(ratings_df['BGGId'].unique())
item_id_map = { orig_item_id : new_item_id for new_item_id, orig_item_id in enumerate(uniq_item_ids) }
n_items = len(uniq_item_ids)

ratings_df = ratings_df.with_columns(BGGId = ratings_df['BGGId'].replace(item_id_map))

In [ ]:
train_idx, test_idx = next(gss.split(ratings_df, None, ratings_df['Username']))
training_df = ratings_df[train_idx]
leftover_df = ratings_df[test_idx]

training_uniq_user_ids = set(training_df['Username'].unique())
training_user_id_map = { orig_user_id : new_user_id for new_user_id, orig_user_id in enumerate(training_uniq_user_ids) }

training_df = training_df.with_columns(Username = training_df['Username'].replace(training_user_id_map))
training_df = training_df.with_columns(pl.col('Username').cast(pl.Int64))

leftover_uniq_user_ids = set(leftover_df['Username'].unique())
leftover_user_id_map = { orig_user_id : new_user_id for new_user_id, orig_user_id in enumerate(leftover_uniq_user_ids) }

leftover_df = leftover_df.with_columns(Username = leftover_df['Username'].replace(leftover_user_id_map))
leftover_df = leftover_df.with_columns(pl.col('Username').cast(pl.Int64))
leftover_df = leftover_df.with_columns(pl.col('Username').is_duplicated())

query_df, test_df = train_test_split(leftover_df, train_size=0.75, stratify=leftover_df['Username'], random_state=42)

## Feature Transformation, Imputation, and Scaling

In [ ]:
training_df = training_df.fill_null(0)
query_df = query_df.fill_null(0)
test_df = test_df.fill_null(0)

In [ ]:
len(training_df.columns)

In [ ]:
# scaler = StandardScaler()
# training_df = scaler.fit_transform(training_df)
# query_df = scaler.fit_transform(query_df)
# test_df = scaler.fit_transform(test_df)

## Model Selection and Evaluation

In [ ]:
"""
HyperParameters
"""
hidden_layer_sizes = [2024]
activation = 'relu'
solver = 'adam'
learning_rate_init = 0.001
random_state = 42

In [ ]:
mlp_base = MLPRegressor(hidden_layer_sizes=hidden_layer_sizes, activation=activation, solver=solver, learning_rate_init=learning_rate_init, random_state=42)
mlp_base.fit(training_df.drop(['Rating', 'Username', 'BGGId']), training_df['Rating'])
# mlp_base_pred = mlp_base.predict(query_df.drop(['Rating', 'Username', 'BGGId']))
# pred_df = query_df.with_columns(PredRating=mlp_1_pred)
# pred_df.head()